# Convolutional Hybrid QNN — FFT Frequency-Domain Fusion (12-Config Ablation)

Replaces the FiLM-based `QuantumGuidedFusion` with a **frequency-domain (FFT) fusion module**:

```
c (B, 64) ──proj_c──► rfft ─┐
                              ├─ product OR sum ─► irfft ─► LayerNorm ─► activation ─► Linear ─► logits
q (B, q_dim) ─proj_q──► rfft ─┘
```

**Configuration axes (2 × 2 × 3 = 12 experiments):**
- `quantum_type`: `real` (PauliZ expvals, dim=4) · `complex` (statevector ℝ+ℑ, dim=32)
- `fusion_op`: `product` (convolution in freq domain) · `sum` (superposition)
- `final_act`: `relu` · `gelu` · `softmax` (applied to fused feature before classifier)

**Baseline:** ConvolutionalHybridQNN with FiLM fusion → 0.9566 macro F1 (10% dataset)

In [1]:
import os
import random
import copy
import json
import time
import math
import itertools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from scipy.signal import get_window
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | Seed: {SEED}')

# ── FFT Fusion Hyper-param ─────────────────────────────────────────────────────
D_FFT = 64  # shared projection dimension for FFT-domain fusion

Device: cuda | Seed: 42


In [2]:
# ==========================================
# DATASET CONFIGURATION  (unchanged)
# ==========================================
BASE_DIR = Path('/home/sammarv/quantum_corrosion')

TARGET_CLASSES = [
    {'id': 0, 'name': 'Spread 0.5g', 'path_sub': 'data/raw/0.5/0.5.iq',  'gram': 0.5},
    {'id': 1, 'name': 'Spread 1.0g', 'path_sub': 'data/raw/1/1.iq',      'gram': 1.0},
    {'id': 2, 'name': 'Spread 1.5g', 'path_sub': 'data/raw/1.5/1.5.iq',  'gram': 1.5},
    {'id': 3, 'name': 'Spread 2.0g', 'path_sub': 'data/raw/2/2.iq',      'gram': 2.0},
    {'id': 4, 'name': 'Spread 2.5g', 'path_sub': 'data/raw/2.5/2.5.iq',  'gram': 2.5},
]

N_CLASSES   = len(TARGET_CLASSES)
LABEL_NAMES = [tc['name'] for tc in TARGET_CLASSES]

OUT_DIR = BASE_DIR / 'results/spread_conv_hybrid_qnn_fft'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FFT_SIZE        = 4096
N_STACKS        = 16
OVERLAP         = 0.25
HOP             = int(FFT_SIZE * (1 - OVERLAP))
SAMPLES_PER_IMG = N_STACKS * HOP + (FFT_SIZE - HOP)
WIN             = get_window('hann', FFT_SIZE).astype(np.float32)

iqs = {}
for tc in TARGET_CLASSES:
    p = BASE_DIR / tc['path_sub']
    iq = np.memmap(str(p), dtype='complex64', mode='r')
    n_imgs = len(iq) // SAMPLES_PER_IMG
    iqs[tc['id']] = iq
    print(f"  label={tc['id']}  {tc['name']:14s}  images={n_imgs:,}")

  label=0  Spread 0.5g     images=9,585
  label=1  Spread 1.0g     images=8,666
  label=2  Spread 1.5g     images=9,141
  label=3  Spread 2.0g     images=10,416
  label=4  Spread 2.5g     images=10,643


In [3]:
# ==========================================
# FEATURE EXTRACTION  (unchanged)
# ==========================================

def extract_sota_features(iq_mmap, img_idx):
    start = img_idx * SAMPLES_PER_IMG
    end   = start + SAMPLES_PER_IMG
    if end > len(iq_mmap):
        return None
    frames = np.empty((N_STACKS, FFT_SIZE), dtype='complex64')
    for i in range(N_STACKS):
        s = start + i * HOP
        frames[i] = iq_mmap[s : s + FFT_SIZE]
    spec     = np.fft.fftshift(np.fft.fft(frames * WIN, axis=1), axes=1)
    mag      = np.abs(spec).astype(np.float32)
    spec_db  = 20.0 * np.log10(mag + 1e-12)
    spec_512 = spec_db.reshape(N_STACKS, 512, 8).mean(axis=2)
    return spec_512


def extract_class(label, n_samples, n_workers=16, desc=''):
    iq     = iqs[label]
    n_imgs = len(iq) // SAMPLES_PER_IMG
    idxs   = np.random.choice(n_imgs, size=n_samples, replace=(n_samples > n_imgs))
    features, labels_out = [], []

    def _worker(idx):
        spec = extract_sota_features(iq, int(idx))
        if spec is None:
            return None, None
        sig_std = spec.std()
        noise   = np.random.randn(*spec.shape).astype(np.float32) * (sig_std * 0.05)
        return spec, spec + noise

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_worker, idx): idx for idx in idxs}
        for fut in as_completed(futs):
            orig, aug = fut.result()
            if orig is not None:
                features.append(orig)
                features.append(aug)
                labels_out.extend([label, label])
            done += 1
            if done % 500 == 0 or done == len(idxs):
                print(f'  [{desc}] {done}/{len(idxs)}  {time.time()-t0:.0f}s')
    return features, labels_out


print('Feature extraction utilities ready.')

Feature extraction utilities ready.


In [4]:
# ==========================================
# BUILD DATASET  (cached, same as original)
# ==========================================
cache_dir = BASE_DIR / 'results/spread_conv_hybrid_qnn/feature_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

DATASET_FRACTION       = 0.10
SAMPLES_PER_CLASS_BASE = 3000
SAMPLES_PER_CLASS      = max(1, int(SAMPLES_PER_CLASS_BASE * DATASET_FRACTION))
fraction_tag           = f"{int(DATASET_FRACTION * 100)}pct"

X_cache = cache_dir / f'X_spec_{fraction_tag}.npy'
y_cache = cache_dir / f'y_{fraction_tag}.npy'

if X_cache.exists() and y_cache.exists():
    print(f'Loading {fraction_tag} features from cache...')
    X_all = np.load(X_cache)
    y_all = np.load(y_cache)
else:
    print('Extracting features...')
    all_feats, all_labels = [], []
    for tc in TARGET_CLASSES:
        feats, lbls = extract_class(tc['id'], SAMPLES_PER_CLASS, desc=tc['name'])
        all_feats.extend(feats)
        all_labels.extend(lbls)
    X_all = np.stack(all_feats).astype(np.float32)
    y_all = np.array(all_labels, dtype=np.int64)
    np.nan_to_num(X_all, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    np.save(X_cache, X_all)
    np.save(y_cache, y_all)

print(f'Dataset: {X_all.shape}  labels: {np.bincount(y_all)}')

Loading 10pct features from cache...
Dataset: (3000, 16, 512)  labels: [600 600 600 600 600]


In [5]:
# ==========================================
# PREPROCESSING & DATA LOADERS  (unchanged)
# ==========================================
N, H, W  = X_all.shape
X_flat   = X_all.reshape(N, -1)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_flat).reshape(N, H, W).astype(np.float32)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_scaled, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)

X_train_t = torch.tensor(X_train_np[:, None, :, :], dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.long)
X_test_t  = torch.tensor(X_test_np[:, None, :, :],  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np,  dtype=torch.long)

batch_size = 32
_n_workers = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size, shuffle=True, drop_last=True,
    num_workers=_n_workers, pin_memory=True, persistent_workers=True,
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=batch_size, shuffle=False,
    num_workers=_n_workers, pin_memory=True, persistent_workers=True,
)

print(f'Train: {X_train_np.shape}  Test: {X_test_np.shape}')
print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

Train: (2400, 16, 512)  Test: (600, 16, 512)
Train batches: 75  Test batches: 19


In [6]:
# ==========================================
# QUANTUM DEVICES & CIRCUITS
# ==========================================
n_qubits = 4
n_layers = 4

def _make_device(n_q, prefer_gpu=True):
    if prefer_gpu:
        try:
            d = qml.device('lightning.gpu', wires=n_q)
            print(f'  {d.name}')
            return d, 'adjoint'
        except Exception:
            pass
    d = qml.device('default.qubit', wires=n_q)
    print(f'  {d.name}')
    return d, 'backprop'

print('Quantum device (real qnode):')
dev_real, diff_real = _make_device(n_qubits)

# Complex statevector always uses default.qubit + backprop for reliable batch state output
print('Quantum device (complex qnode):')
dev_complex = qml.device('default.qubit', wires=n_qubits)


# ── Real qnode: PauliZ expectation values (original) ──────────────────────────
@qml.qnode(dev_real, interface='torch', diff_method=diff_real)
def qnode_real(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# ── Complex qnode: full statevector (2^n_qubits complex amplitudes) ────────────
# Returns the quantum state as complex amplitudes, preserving both
# magnitude (Born probabilities) and phase (quantum interference) information.
@qml.qnode(dev_complex, interface='torch', diff_method='backprop')
def qnode_complex(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.state()  # (2^n_qubits,) complex — 16 complex amplitudes for n_qubits=4


print(f'\nn_qubits={n_qubits}  n_layers={n_layers}')
print(f'Real output dim : {n_qubits}')
print(f'Complex output dim: 2 * 2^{n_qubits} = {2 * 2**n_qubits}  (real+imag split)')

Quantum device (real qnode):
  Default qubit PennyLane plugin
Quantum device (complex qnode):

n_qubits=4  n_layers=4
Real output dim : 4
Complex output dim: 2 * 2^4 = 32  (real+imag split)


In [7]:
# ==========================================
# BATCHED QUANTUM LAYERS
# ==========================================

class BatchedQuantumLayer(nn.Module):
    """Original batched PauliZ expval layer (unchanged from baseline).

    Calling qnode with (B, n_qubits) triggers default.qubit batch broadcasting.
    Batched output: (n_qubits * B,) interleaved by qubit → reshape to (B, n_qubits).
    """
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits = n_q
        self.qnode    = q_node
        self.weights  = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))

    def forward(self, x):
        B   = x.shape[0]
        out = self.qnode(x, self.weights)          # (n_qubits * B,) interleaved
        return out.float().view(self.n_qubits, B).t().contiguous()  # (B, n_qubits)


class BatchedQuantumLayerComplex(nn.Module):
    """Complex statevector layer: returns real + imaginary amplitudes concatenated.

    qnode_complex returns qml.state() → (B, 2^n_qubits) complex under batch broadcasting.
    We split into real and imaginary parts and concatenate → (B, 2 * 2^n_qubits) float.
    This preserves both probability amplitudes and quantum phase information.
    """
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits   = n_q
        self.state_dim  = 2 ** n_q   # 16 for n_qubits=4
        self.qnode      = q_node
        self.weights    = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))

    def forward(self, x):
        B   = x.shape[0]
        out = self.qnode(x, self.weights)   # (B, 2^n_qubits) complex
        # Normalise shape defensively: PennyLane batch state shape should be (B, state_dim)
        if out.dim() == 1:
            out = out.unsqueeze(0)
        if out.shape[0] != B:
            out = out.t()                   # fallback if axes are swapped
        q_real = out.real.float()           # (B, 16)
        q_imag = out.imag.float()           # (B, 16)
        return torch.cat([q_real, q_imag], dim=-1).contiguous()  # (B, 32)

In [8]:
# ==========================================
# FREQUENCY-DOMAIN FUSION MODULE
# ==========================================

class FreqDomainFusion(nn.Module):
    """
    Quantum-Classical fusion in the frequency domain.

    Pipeline:
      c (B, c_dim) ──proj_c──► rfft ─┐
                                      ├─ fusion_op ─► irfft ─► LayerNorm ─► act ─► classifier
      q (B, q_dim) ──proj_q──► rfft ─┘

    fusion_op='product' : complex element-wise multiplication in freq domain
                          (equivalent to cross-correlation in spatial domain)
    fusion_op='sum'     : complex element-wise addition (linear superposition)

    final_act           : non-linearity applied to the recovered spatial feature
                          before the linear classifier (logits always raw for CE loss)
    """
    def __init__(self, c_dim, q_dim, d_fft, n_classes,
                 fusion_op='product', final_act='gelu'):
        super().__init__()
        self.d_fft     = d_fft
        self.fusion_op = fusion_op

        self.proj_c = nn.Linear(c_dim, d_fft)
        self.proj_q = nn.Linear(q_dim, d_fft)
        self.norm   = nn.LayerNorm(d_fft)

        _acts = {'relu': nn.ReLU(), 'gelu': nn.GELU(), 'softmax': nn.Softmax(dim=-1)}
        if final_act not in _acts:
            raise ValueError(f'final_act must be one of {list(_acts)}')
        self.act = _acts[final_act]

        self.classifier = nn.Linear(d_fft, n_classes)

    def forward(self, c, q):
        c_proj = self.proj_c(c)                            # (B, d_fft)       real
        q_proj = self.proj_q(q)                            # (B, d_fft)       real

        C = torch.fft.rfft(c_proj, dim=-1)                 # (B, d_fft//2+1)  complex
        Q = torch.fft.rfft(q_proj, dim=-1)                 # (B, d_fft//2+1)  complex

        if self.fusion_op == 'product':
            F = C * Q                                       # pointwise complex multiplication
        else:  # 'sum'
            F = C + Q                                       # pointwise complex addition

        f = torch.fft.irfft(F, n=self.d_fft, dim=-1)       # (B, d_fft)       real
        f = self.norm(f)                                    # stabilise magnitude
        f = self.act(f)                                     # shape feature space
        return self.classifier(f)                           # (B, n_classes)   raw logits

In [9]:
# ==========================================
# HYBRID QNN WITH FFT FUSION
# ==========================================

class HybridQNN_FFT(nn.Module):
    """
    Input : (B, 1, 16, 512)  — normalised log-magnitude spectrogram
    Output: (B, N_CLASSES)   — raw logits

    CNN backbone and classical skip are identical to ConvolutionalHybridQNN.
    QuantumGuidedFusion (FiLM) is replaced by FreqDomainFusion (FFT).

    quantum_type : 'real'    → PauliZ expvals (B, 4)
                   'complex' → statevector ℝ+ℑ (B, 32)
    fusion_op    : 'product' | 'sum'
    final_act    : 'relu' | 'gelu' | 'softmax'
    """
    def __init__(self, n_cls=N_CLASSES, dropout_cnn=0.2350, dropout_skip=0.2012,
                 quantum_type='real', fusion_op='product', final_act='gelu',
                 d_fft=D_FFT):
        super().__init__()
        self.quantum_type = quantum_type

        # ── CNN feature extractor (unchanged) ─────────────────────────────────
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),           # → (16, 8, 256)
            nn.Conv2d(16, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),           # → (32, 4, 128)
            nn.Flatten(),
            nn.Linear(32 * 4 * 128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_cnn),
        )

        # ── Quantum branch ─────────────────────────────────────────────────────
        self.qnn_proj = nn.Sequential(
            nn.Linear(256, n_qubits),
            nn.Sigmoid(),
        )
        if quantum_type == 'complex':
            self.qnn  = BatchedQuantumLayerComplex(n_layers, n_qubits, qnode_complex)
            q_dim     = 2 * (2 ** n_qubits)   # 32
        else:
            self.qnn  = BatchedQuantumLayer(n_layers, n_qubits, qnode_real)
            q_dim     = n_qubits               # 4

        # ── Classical skip branch (unchanged) ─────────────────────────────────
        self.classical_skip = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_skip),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_skip),
        )

        # ── FFT fusion (replaces QuantumGuidedFusion) ──────────────────────────
        self.fusion = FreqDomainFusion(
            c_dim=64, q_dim=q_dim, d_fft=d_fft,
            n_classes=n_cls, fusion_op=fusion_op, final_act=final_act,
        )

    def forward(self, x):
        features  = self.cnn(x)
        qbn_input = self.qnn_proj(features) * (2.0 * np.pi)
        q_feats   = self.qnn(qbn_input)
        c_feats   = self.classical_skip(features)
        return self.fusion(c_feats, q_feats)

In [10]:
# ==========================================
# SMOKE TEST — verify all 4 leaf configurations
# ==========================================
print('Smoke testing all quantum_type × fusion_op combinations...\n')
_x = X_train_t[:2].to(device)

for qt in ('real', 'complex'):
    for fo in ('product', 'sum'):
        m = HybridQNN_FFT(quantum_type=qt, fusion_op=fo, final_act='gelu').to(device)
        with torch.no_grad():
            out = m(_x)
        n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
        print(f'  quantum={qt:7s}  fusion={fo:7s}  out={list(out.shape)}  params={n_p:,}')
        del m

print('\nAll smoke tests passed.')

Smoke testing all quantum_type × fusion_op combinations...

  quantum=real     fusion=product  out=[2, 5]  params=4,247,097
  quantum=real     fusion=sum      out=[2, 5]  params=4,247,097
  quantum=complex  fusion=product  out=[2, 5]  params=4,248,889
  quantum=complex  fusion=sum      out=[2, 5]  params=4,248,889

All smoke tests passed.


In [11]:
# ==========================================
# TRAINING HYPERPARAMETERS  (same as baseline)
# ==========================================
lr           = 0.00079146
weight_decay = 1.11589e-06

class_counts = np.bincount(y_train_np)
cw_vals      = 1.0 / class_counts.astype(np.float32)
cw_vals      = cw_vals / cw_vals.sum() * N_CLASSES
loss_weights = torch.tensor(cw_vals, dtype=torch.float32).to(device)

# Per-config budget: shorter than the full 100-epoch baseline to keep
# the 12-config sweep tractable while still capturing convergence trends.
MAX_EPOCHS_PER_CONFIG = 30
PATIENCE_PER_CONFIG   = 5

print(f'lr={lr}  wd={weight_decay}')
print(f'class weights: {cw_vals.tolist()}')
print(f'Per-config budget: {MAX_EPOCHS_PER_CONFIG} epochs, patience={PATIENCE_PER_CONFIG}')

lr=0.00079146  wd=1.11589e-06
class weights: [1.0, 1.0, 1.0, 1.0, 1.0]
Per-config budget: 30 epochs, patience=5


In [12]:
# ==========================================
# TRAINING UTILITY
# ==========================================

def train_config(quantum_type, fusion_op, final_act,
                 n_epochs=MAX_EPOCHS_PER_CONFIG, patience=PATIENCE_PER_CONFIG,
                 verbose=True):
    """Train one HybridQNN_FFT configuration and return test-set metrics."""
    set_seed(SEED)
    model     = HybridQNN_FFT(quantum_type=quantum_type,
                               fusion_op=fusion_op,
                               final_act=final_act).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(weight=loss_weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_f1       = 0.0
    best_weights  = None
    best_epoch    = 0
    no_impr       = 0

    for epoch in range(n_epochs):
        # ── train ────────────────────────────────────────────────────────────
        model.train()
        for inputs, targets in train_loader:
            inputs  = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()

        # ── validate ─────────────────────────────────────────────────────────
        model.eval()
        all_preds, all_tgts = [], []
        with torch.no_grad():
            for inputs, targets in test_loader:
                out = model(inputs.to(device, non_blocking=True))
                all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                all_tgts.extend(targets.numpy())

        f1 = f1_score(all_tgts, all_preds, average='macro', zero_division=0)
        scheduler.step(f1)

        if f1 > best_f1:
            best_f1      = f1
            best_weights = copy.deepcopy(model.state_dict())
            best_epoch   = epoch + 1
            no_impr      = 0
        else:
            no_impr += 1

        if verbose:
            mark = ' ←' if no_impr == 0 else ''
            print(f'  ep {epoch+1:02d}  f1={f1:.4f}{mark}')

        if no_impr >= patience:
            break

    # ── final evaluation with best checkpoint ─────────────────────────────────
    model.load_state_dict(best_weights)
    model.eval()
    all_preds, all_tgts = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            out = model(inputs.to(device))
            all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            all_tgts.extend(targets.numpy())

    return {
        'val_acc':      accuracy_score(all_tgts, all_preds),
        'macro_f1':     f1_score(all_tgts, all_preds, average='macro',    zero_division=0),
        'weighted_f1':  f1_score(all_tgts, all_preds, average='weighted', zero_division=0),
        'best_epoch':   best_epoch,
        'total_epochs': epoch + 1,
    }

In [13]:
# ==========================================
# 12-CONFIG SWEEP
# ==========================================
QUANTUM_TYPES = ['real', 'complex']
FUSION_OPS    = ['product', 'sum']
FINAL_ACTS    = ['relu', 'gelu', 'softmax']

CONFIGS = list(itertools.product(QUANTUM_TYPES, FUSION_OPS, FINAL_ACTS))
print(f'Running {len(CONFIGS)} configurations...\n')
print(f'  Quantum types : {QUANTUM_TYPES}')
print(f'  Fusion ops    : {FUSION_OPS}')
print(f'  Final acts    : {FINAL_ACTS}')
print(f'  Epochs budget : {MAX_EPOCHS_PER_CONFIG} / patience {PATIENCE_PER_CONFIG}')
print(f'  Baseline F1   : 0.9566 (FiLM fusion, PauliZ expvals)\n')

results = []
t_total = time.time()

for i, (qt, fo, fa) in enumerate(CONFIGS, 1):
    label = f'[{i:02d}/{len(CONFIGS)}]  quantum={qt:7s}  fusion={fo:7s}  act={fa:7s}'
    print('=' * 65)
    print(label)
    print('=' * 65)
    t0 = time.time()

    metrics = train_config(qt, fo, fa, verbose=True)

    elapsed = time.time() - t0
    print(f'  → val_acc={metrics["val_acc"]:.4f}  '
          f'macro_f1={metrics["macro_f1"]:.4f}  '
          f'best_epoch={metrics["best_epoch"]}  '
          f'({elapsed:.0f}s)\n')

    results.append({
        'quantum_type': qt,
        'fusion_op':    fo,
        'final_act':    fa,
        **metrics,
    })

print(f'Sweep complete in {time.time() - t_total:.0f}s')

Running 12 configurations...

  Quantum types : ['real', 'complex']
  Fusion ops    : ['product', 'sum']
  Final acts    : ['relu', 'gelu', 'softmax']
  Epochs budget : 30 / patience 5
  Baseline F1   : 0.9566 (FiLM fusion, PauliZ expvals)

[01/12]  quantum=real     fusion=product  act=relu   
  ep 01  f1=0.8460 ←
  ep 02  f1=0.9237 ←
  ep 03  f1=0.9583 ←
  ep 04  f1=0.9299
  ep 05  f1=0.9400
  ep 06  f1=0.9501
  ep 07  f1=0.9401
  ep 08  f1=0.9466
  → val_acc=0.9583  macro_f1=0.9583  best_epoch=3  (72s)

[02/12]  quantum=real     fusion=product  act=gelu   
  ep 01  f1=0.8719 ←
  ep 02  f1=0.9097 ←
  ep 03  f1=0.9314 ←
  ep 04  f1=0.9364 ←
  ep 05  f1=0.9366 ←
  ep 06  f1=0.9533 ←
  ep 07  f1=0.9433
  ep 08  f1=0.9550 ←
  ep 09  f1=0.9530
  ep 10  f1=0.9616 ←
  ep 11  f1=0.9650 ←
  ep 12  f1=0.9533
  ep 13  f1=0.9550
  ep 14  f1=0.9599
  ep 15  f1=0.9432
  ep 16  f1=0.9533
  → val_acc=0.9650  macro_f1=0.9650  best_epoch=11  (143s)

[03/12]  quantum=real     fusion=product  act=softmax

In [14]:
# ==========================================
# RESULTS TABLE
# ==========================================
df = pd.DataFrame(results)
df = df.sort_values('macro_f1', ascending=False).reset_index(drop=True)
df.index = df.index + 1   # 1-based ranking

# Format floats
fmt = {'val_acc': '{:.4f}', 'macro_f1': '{:.4f}', 'weighted_f1': '{:.4f}'}
df_disp = df.copy()
for col, f in fmt.items():
    df_disp[col] = df[col].apply(lambda v: f.format(v))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

BASELINE_F1 = 0.9566

print('\n' + '=' * 80)
print('FFT FUSION — CONFIGURATION COMPARISON  (sorted by Macro F1, best → worst)')
print('=' * 80)
print(df_disp[['quantum_type', 'fusion_op', 'final_act',
               'val_acc', 'macro_f1', 'weighted_f1',
               'best_epoch', 'total_epochs']].to_string())

best    = df.iloc[0]
worst   = df.iloc[-1]
vs_base = best['macro_f1'] - BASELINE_F1

print('\n' + '-' * 80)
print(f'Baseline (FiLM, PauliZ expvals)  → macro F1 = {BASELINE_F1:.4f}')
print(f'Best FFT config                  → macro F1 = {best["macro_f1"]:.4f}'
      f'  ({vs_base:+.4f} vs baseline)')
print(f'  quantum={best["quantum_type"]}  fusion={best["fusion_op"]}  act={best["final_act"]}')
print(f'Worst FFT config                 → macro F1 = {worst["macro_f1"]:.4f}')
print(f'  quantum={worst["quantum_type"]}  fusion={worst["fusion_op"]}  act={worst["final_act"]}')

# Save results
df.to_csv(OUT_DIR / 'fft_fusion_results.csv', index_label='rank')
print(f'\nSaved: {OUT_DIR}/fft_fusion_results.csv')

# ── Grouped summary ──────────────────────────────────────────────────────────
print('\n--- Macro F1 by quantum_type ---')
print(df.groupby('quantum_type')['macro_f1'].agg(['mean', 'max', 'min']).round(4).to_string())

print('\n--- Macro F1 by fusion_op ---')
print(df.groupby('fusion_op')['macro_f1'].agg(['mean', 'max', 'min']).round(4).to_string())

print('\n--- Macro F1 by final_act ---')
print(df.groupby('final_act')['macro_f1'].agg(['mean', 'max', 'min']).round(4).to_string())


FFT FUSION — CONFIGURATION COMPARISON  (sorted by Macro F1, best → worst)
   quantum_type fusion_op final_act val_acc macro_f1 weighted_f1  best_epoch  total_epochs
1       complex   product      gelu  0.9717   0.9717      0.9717          12            17
2          real   product      gelu  0.9650   0.9650      0.9650          11            16
3       complex   product      relu  0.9617   0.9617      0.9617          15            20
4          real   product      relu  0.9583   0.9583      0.9583           3             8
5          real       sum      relu  0.9567   0.9567      0.9567          10            15
6          real       sum      gelu  0.9567   0.9566      0.9566          15            20
7       complex       sum      gelu  0.9567   0.9566      0.9566          12            17
8       complex       sum      relu  0.9500   0.9500      0.9500          18            23
9          real   product   softmax  0.6000   0.4692      0.4692          19            24
10         real